### Implementing Regularization with Linear Regression:
*
- *How to choose the right `alpha` for the data and question at hand?*

    - `Lasso(alpha = 0.5)` :
    - `Ridge(alpha = 0.5)` :

##### **EXAMPLE**

```python
# EDA:
import pandas as pd

df = pd.read_cvs('student_math.cvs')
print(df.columns, df.shape)
```

---

```python
# Set predictor and outcome variables and perform a train-test-split:

y = df['Final_Grade']
X = df.drop(columns = ['Final_Grade'])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test  = train_test_split(X, y, test_size=0.33, random_state=42)
```

---

```python
# Fit Lasso regularization regression model:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.05) # hyperparameter that gets tuned
lasso.fit(X_train, y_train)
```

---

```python
# Look over Mean Squared Error (MSE):
from sklearn.metrics import mean_squared_error

pred_train = lasso.predict(X_train)
pred_test = lasso.predict(X_test)

training_mse = mean_squared_error(y_train, y_test)
test_mes = mean_squared_error(y_test, pred_test)

print('Training Error: ',  training_mse)
print('Test Error: ', test_mse)
```
- *Output indicates that the model is still overfitting the data and that there is room to regularize more.*

---


#### Tuning Regularization Hyperparameter:

- `scikit-learn` default `alpha` is 1.0
    - an `alpha` of zero is equivalent to no regularization.
    - an `alpha` too big has a problem of biasing the model, and underfitting the data.

- Iterate over an array of `alpha` values and plot the resulting training and test errors against the corresponding `alpha`'s. 

![alpha tuning](images/alpha_tuning.png)

### Grid Search Cross-validation:

- `GridSearchCV` : Grid Search Cross-validation
    - helps to satisfactory resolve `alpha` tuning while automating and speeding up our hyperparameter search.

**Automate Hyperparameter Search with `GridSearchCV`**

- `GridSearchCV` uses k-fold cross-validation method to search for the optimal hyperparameter ina machine learning algorithm.

    - The k-fold method iteratively splits the dataset into train-test splits “k” times such that every point in the sample gets to be within the test data at least once.

    - Doing a k-fold cross-validation makes sure that the conclusions drawn about the data are not the result of a sampling effect but are truly representative of all of the data.

---

- `GridSearchCV` arguments:

    - ***estimator*** :the machine learning algorithm whose hyperparameters are being tuned. 
        - `Lasso()` or `Ridge()`

    - ***param_grid*** :a grid of potential values for the hyperparameters that are being tuned.
        - This has to be in the form of a dictionary with the keys representing the parameter inputs to the model and the values, lists of potential parameter values.

        - *example* :Lasso implementations only have one hyperparameter for tuning : `alpha` 


- `logspace` function:

    - Similar to `linspace`, but gets numbers that are evenly spaced in the logarithmic scale. 
        - takes in the powers of tens that we're searching between. 

    ```python
    import numpy as np

    ## an array of alpha values between 0.000001 and 1.0
    alpha_array = np.logspace(-6, 0, 100)

    #dict with key (alpha) and values being alpha_array
    tuned_parameters = [{'alpha': alpha_array}]
    ```
    - **scoring**: the metric used to evaluate the performance of the model on the test set. `GridSearchCV` searches for the set of parameters within param_grid that maximizes this value. 

*scoring strategy is the argument for mean squared error*

- `cv`: specifies the way the cross-validation is performed. The default here is the 5-fold cross-validation method that’s described above. Otherwise one can set a different number of folds (i.e., an alternate positive integer value for k)

- `return_train_score`: a Boolean argument specifying whether we would like the output of our fit to return the scores on training data every folds The default here is False but we’re going to set it to True as we’re using GridSearchCV to correct for overfitting and would like to be kept in the loop about the training score.

- to check the result of implementing different values of `alpha` and to do multiple train-test splits to make sure we’re covering the entire sample:

```python
from sklearn.model_selection import GridSearchCV

model = GridSearchCV(estimator = Lasso(), param_grid = tuned_parameters, scoring = 'neg_mean_squared_error', cv=5, return_train_score=True)

model.fit(X,y)
```

---

- Examine Attributes:

    - `cv_results_` : object gives us the details of every model fit corresponding to a particular `alpha` value and train-test split of each fold. 
    - (Example: 100 values with a 5-fold cross-validated search = 500 model fits)

```python

test_scores = model.cv_results_['mean_test_score']
train_scores = model.cv_results_['mean_train_score']
```

*objects are an array the size of param_grid; the number of `alpha` values specified*

- To get the `alpha` optimal to our scoring strategy:

```python
print(model.best_params_, model.best_score_)
```

![output](images/output_image.png)

> *(Note that the score is the negative of least test mean squared error!)*

![alpha graph image](images/alpha_parameter_tuned.png)

- The blue line here is our tuned hyperparameter and this value of alpha (~0.1233) corresponds to the optimal cross-validated test and training error values.

---

> *Note: The alpha values are orders of magnitude bigger for Ridge for the following reason: alpha is inversely proportional to the size of the regularization constraint. Recall that for Lasso this means that it’s proportional to `1/s` while for Ridge, this would be `1/s^2` as the constraint surfaces are different. So reducing s by a factor of 10 means increasing alpha 10x for Lasso but 100x for Ridge.*



## Regularization with Logistic Regression Classifier:

- scikit-learn logistic regression implementation is regularized by default. L2-regularize with an `alpha` of 100. 

- Attributes of a logistic regression model:
    - `penalty` : options - 'l1', 'l2', 'none', 'elasticnet' - default = 'l2'

    - `C` : the inverse of regularization strength: default value = 1.0

    - `solver` : the options are {'lbfgs', liblinear', 'newton_cg', 'sag', 'saga'}

- `LogisticRegression()` function differs from `Lasso()` and `Ridge()` functions for Linear Regression in that its input is the parameter `C`, which is the inverse of `alpha`.

> *This is important to keep in mind, especially while setting up the parameter grid prior to implementing `GridSearchCV`.*

---

- **Lasso (L1)**: L1 implementation requires setting the penalty attribute to ‘l1’ and also setting the solver attribute to ‘liblinear’

```python 
logistic_lasso = LosgisticRegression(penalty='l1', solver='liblinear', C= ___)
```

> *there is a solver attribute in scikit-learn for most machine learning algorithms. The default solver for `LogisticRegression()` is `lbfgs`, however the only solver that can be used with Lasso Regularization is the `‘liblinear’` solver and hence it needs to be explicitly specified*

---

- **Ridge (L2)**: L2 is default, so only `C` needs to be specified

```python
logistic_ridge = LogisticRegression(C= ___)
```

---

- **Elasticnet**: 

    - Elasticnet Regulariztion is a coombination of L1 and L2 regularization. Has two penalty terms, one for L1 and L2, respectfully.

penalty options:
    - 'l1'
    - 'l2'
    - 'none'

```python
logistic_elasticnet = LogisticRegression(penalty='elasticnet', solver='saga', C= ___, l1_ratio= ___)
```

- Implementation of Elasticnet Regularization:
    - need two hyperparameters: 

        - `C` : specifies regularization strength

        - `l1_ratio` : mixing hyperparameter that specifies how much L1 regularization used relative to L2
            - takes values between 0 and 1; 1 being equivalent to applying just L1, 0 being equivalent to applying just L2 penalties
        
        - `saga` : solver

---

### Tuning Hyperparameter `C`: (With `GridSearchCV` and `LogisiticRegressionCV`):

> *Remember that C is the inverse of alpha so greater the C, the lesser the amount of regularization*


**GridSearchCv**

```python
#Making an array of C's; here we're choosing 100 values between 0.001 and 100
C_array  = np.logspace(-3,2, 100)

#Making a dict to enter as an input to param_grid
tuning_C = {'C':C_array}
clf = LogisticRegression(penalty = 'l1', solver =  'liblinear')
gs = GridSearchCV(clf, param_grid = tuning_C, scoring = 'accuracy', cv = 5)
gs.fit(X,y) #remember to fit the model before printing

print(gs.best_params_)
print(gs.best_score_)

# OUTPUT:
# {'C': 2.420128264794381}
# 0.8823529411764705
```

---

**LogisticRegressionCV()**

```python
model = LogisticRegressionCV(Cs=np.logspace(-3,2, 100), penalty='l2', scoring='accuracy', cv=5, random_state=42,max_iter=10000)
model.fit(X, y)
print(model.C_, model.scores_[1].mean(axis=0).max())

# OUTPUT:
# [2.42012826] 0.8823529411764705
```

> Research: scikit-learn cross-validation fuctions: `LassoCV()`, `RidgeCV()`, `ElasticnetCV()`
